# 09 — DiffusionGemma in vLLM: stock vs. top-k self-conditioning, end to end

**Question:** inside a real serving engine, with the whole model running, how do throughput, latency, accuracy and the largest batch that fits change when the self-conditioning hint uses only the top-k tokens?

**How the change gets into vLLM.** The folder `dg_topk_plugin/` is a small **vLLM plugin**. vLLM loads plugins in every one of its processes (including the tensor-parallel workers). Ours does nothing unless the environment variable `DG_TOPK_SC_K` is set; with `DG_TOPK_SC_K=64` it replaces two functions of vLLM's DiffusionGemma sampler:

| vLLM function | stock | with the plugin |
|---|---|---|
| `sample_row_stats` | one fused pass (argmax, sample, entropy) + a second pass writing the full softmax in bf16 | the same first pass (identical decisions for the same seed), **no softmax write**; returns the top-k ids and probabilities |
| `_store_self_conditioning` | `probs @ embedding_table` over all 262,144 tokens | an `embedding_bag` over the k rows; under tensor parallelism each GPU sums its shard and the results are all-reduced, as in vLLM |

So the commit, re-noise and stop decisions stay vLLM's own; only the self-conditioning input changes. The third configuration, `topk64_exact`, also sets `DG_TOPK_SAMPLE=topk_exact`: no per-logit random numbers, the token is drawn exactly from the top-k with a rare tail branch (same distribution, different random numbers, so outputs are not token-identical to stock).

**What this notebook does**
1. Checks the environment (vLLM version with DiffusionGemma, the plugin, 2 free GPUs).
2. For each configuration, starts **one** vLLM server with the largest batch limit (`--max-num-seqs`) that fits (trying the largest first; a failure to start is recorded), then sends GSM8K questions at each concurrency level up to that limit and records throughput, latency and accuracy. One server per configuration keeps the GPUs reserved for the whole sweep on a shared machine and saves model loads.
3. Compares stock vs. top-k: tokens/s, latency, accuracy, the largest `--max-num-seqs` that fits, and the KV-cache memory vLLM reports at startup (vLLM sizes the KV cache after a profiling run that includes the sampler, so a lighter sampler leaves more memory for the cache).
4. **Kernel check (last, so this notebook's own GPU use cannot take memory from the servers):** vLLM's own `sample_row_stats` vs. the copy used by notebook 08 and the plugin: same results for the same seed, similar speed.

Run it in the `.venv-vllm` environment (RUNBOOK.md, sections 1b and 3). It needs **both A100s free**.

In [ ]:
import os, sys, time, json, re, signal, shutil, subprocess, importlib.metadata as md_
import urllib.request

MODEL_ID = 'google/diffusiongemma-26B-A4B-it'
N_GPUS = 2                        # tensor-parallel size: 2 or 4 (4 leaves far more memory for large batches)
GPUS = None                       # e.g. '0,1'; None = the N_GPUS GPUs with the most free memory
PORT = 8000
N_QUESTIONS = 100                 # GSM8K questions per (config, batch limit); raise to 250+ for final numbers
MAX_SEQS = [1, 2, 4, 8, 16] + ([32, 64] if N_GPUS >= 4 else [])   # --max-num-seqs values; vLLM's recipe needs 4 on one 80 GB GPU in bf16
CONFIGS = {
    'stock':        {},
    'topk64':       {'DG_TOPK_SC_K': '64', 'DG_TOPK_SC_TAIL': 'renorm'},                        # fused top-k hint, vLLM's sampling
    'topk64_exact': {'DG_TOPK_SC_K': '64', 'DG_TOPK_SC_TAIL': 'renorm', 'DG_TOPK_SAMPLE': 'topk_exact'},  # + exact top-k sampling
}
SERVER_ARGS = ['--tensor-parallel-size', str(N_GPUS), '--max-model-len', '4096', '--attention-backend', 'TRITON_ATTN',
               '--enforce-eager']   # flags from vLLM's own DiffusionGemma GSM8K test (memory utilization set below)

# GPUs: the machine is shared, so pick the two with the most free memory and give vLLM only what is free.
rows = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.free,memory.total', '--format=csv,noheader,nounits'],
                      capture_output=True, text=True, check=True).stdout.strip().splitlines()
gpu_free = {int(i): (int(f) / 1024, int(t) / 1024) for i, f, t in (r.split(', ') for r in rows)}
if GPUS is None:
    GPUS = ','.join(str(i) for i, _ in sorted(gpu_free.items(), key=lambda kv: -kv[1][0])[:N_GPUS])
chosen = [int(g) for g in GPUS.split(',')]
assert len(chosen) == N_GPUS, f'GPUS lists {len(chosen)} GPUs but N_GPUS = {N_GPUS}'
min_free = min(gpu_free[g][0] for g in chosen)
total = gpu_free[chosen[0]][1]
GPU_UTIL = round(min(0.92, (min_free - 1.0) / total), 2)
print(f'using GPUs {GPUS} | free GB {[round(gpu_free[g][0], 1) for g in chosen]} | --gpu-memory-utilization {GPU_UTIL}')
weights_per_gpu = 52 / N_GPUS
if GPU_UTIL * total < weights_per_gpu + 6:
    print(f'WARNING: less than ~{weights_per_gpu + 6:.0f} GB per GPU for vLLM; bf16 weights alone take ~{weights_per_gpu:.0f} GB per GPU at tensor parallel {N_GPUS}. '
          'Servers will likely fail to start. Free the GPUs first.')
SERVER_ARGS += ['--gpu-memory-utilization', str(GPU_UTIL)]
OUT = './runs/nb09'
os.makedirs(OUT, exist_ok=True)

# --- checks. This process never uses the GPUs for the server runs (the kernel check runs at the end),
# so the servers get all the free memory.
import vllm
print('python', sys.executable)
print('vllm', vllm.__version__)
try:
    from vllm.model_executor.models.diffusion_gemma_sampler import sample_row_stats  # noqa: F401
    print('DiffusionGemma sampler: found')
except Exception as e:
    raise SystemExit(f'This vLLM has no DiffusionGemma sampler ({e}). Install a newer vLLM (RUNBOOK section 1b).')
eps = [e for e in md_.entry_points(group='vllm.general_plugins') if e.name == 'dg_topk']
print('dg_topk plugin installed:', bool(eps), '| install with: pip install -e ./dg_topk_plugin' if not eps else '')
assert eps, 'plugin not installed'
print(subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.free', '--format=csv'], capture_output=True, text=True).stdout)

## Server and client helpers

In [ ]:
VLLM_BIN = shutil.which('vllm') or os.path.join(os.path.dirname(sys.executable), 'vllm')

def start_server(name, extra_env, max_seqs, timeout_s=1500):
    # Start `vllm serve` in the background; return (process, log path) once /health answers, or (None, log) on failure.
    log_path = f'{OUT}/server_{name}_seqs{max_seqs}.log'
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPUS, **extra_env)
    for key in ('DG_TOPK_SC_K', 'DG_TOPK_SC_TAIL', 'DG_TOPK_SAMPLE'):
        if key not in extra_env:
            env.pop(key, None)
    cmd = [VLLM_BIN, 'serve', MODEL_ID, '--port', str(PORT), '--max-num-seqs', str(max_seqs), *SERVER_ARGS]
    log = open(log_path, 'w')
    proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            return None, log_path                      # exited: usually out of memory, see the log
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=5) as r:
                if r.status == 200:
                    print(f'  server up in {time.time() - t0:.0f}s')
                    return proc, log_path
        except Exception:
            pass
        time.sleep(10)
    stop_server(proc)
    return None, log_path

def stop_server(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=120)
    except Exception:
        os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(15)                                     # let the GPUs free their memory

def tail(path, n=15):
    return ''.join(open(path).readlines()[-n:])

def kv_cache_stats(path):
    # vLLM logs the KV-cache memory and size it settled on after its profiling run.
    text = open(path).read()
    mem = re.findall(r'Available KV cache memory:\s*([\d.]+)\s*GiB', text)
    tok = re.findall(r'GPU KV cache size:\s*([\d,]+)\s*tokens', text)
    return dict(kv_cache_gib=float(mem[-1]) if mem else float('nan'),
                kv_cache_tokens=int(tok[-1].replace(',', '')) if tok else float('nan'))

In [ ]:
import asyncio
from openai import AsyncOpenAI
from datasets import load_dataset

gsm = load_dataset('openai/gsm8k', 'main', split=f'test[:{N_QUESTIONS}]')

def last_number(s):
    nums = re.findall(r'-?\d[\d,]*\.?\d*', s or '')
    return nums[-1].replace(',', '').rstrip('.') if nums else None

def correct(pred, ans):
    gold = ans.split('####')[-1].strip().replace(',', '')
    try:
        return pred is not None and abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

async def run_client(concurrency):
    # Send every question with at most `concurrency` requests in flight; measure tokens/s, latency, accuracy.
    client = AsyncOpenAI(base_url=f'http://127.0.0.1:{PORT}/v1', api_key='none')
    sem = asyncio.Semaphore(concurrency)
    lat, toks, ok = [], [], []

    async def one(ex):
        async with sem:
            t0 = time.perf_counter()
            r = await client.chat.completions.create(
                model=MODEL_ID, max_tokens=512,
                messages=[{'role': 'user', 'content': ex['question'] + '\nSolve step by step, then give the final answer as a number.'}])
            lat.append(time.perf_counter() - t0)
            toks.append(r.usage.completion_tokens)
            ok.append(correct(last_number(r.choices[0].message.content), ex['answer']))

    t0 = time.perf_counter()
    await asyncio.gather(*(one(ex) for ex in gsm))
    wall = time.perf_counter() - t0
    lat.sort()
    return dict(wall_s=wall, tokens=sum(toks), tok_per_s=sum(toks) / wall, req_per_s=len(gsm) / wall,
                lat_mean_s=sum(lat) / len(lat), lat_p50_s=lat[len(lat) // 2], lat_p90_s=lat[int(len(lat) * 0.9)],
                accuracy=sum(ok) / len(ok), n=len(ok))

## Run: one server per configuration, concurrency sweep on the client side

For each configuration: try `--max-num-seqs` from the largest value down until a server starts (each failed start is recorded: that is the memory result), then send all questions at each concurrency level up to that limit, then stop the server. Rows go to `runs/nb09/e2e.csv` as they finish; a rerun skips (config, concurrency) pairs already done.


In [ ]:
import pandas as pd
CSV = f'{OUT}/e2e.csv'
done = set()
if os.path.exists(CSV):
    prev = pd.read_csv(CSV)
    if 'concurrency' in prev:
        done = set(zip(prev.config, prev.concurrency.fillna(-1).astype(int)))
    print('resuming; already done:', sorted(done))

def append(row):
    pd.DataFrame([row]).to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)

for name, extra_env in CONFIGS.items():
    todo = [c for c in MAX_SEQS if (name, c) not in done]
    if not todo:
        continue
    # 1) the largest --max-num-seqs that starts (the memory result)
    proc = None
    for m in sorted(MAX_SEQS, reverse=True):
        print(f'{name}: starting server with --max-num-seqs {m}')
        t_start = time.time()
        proc, log_path = start_server(name, extra_env, m)
        if proc is not None:
            server_max = m
            info = dict(startup_s=time.time() - t_start, **kv_cache_stats(log_path))
            if extra_env.get('DG_TOPK_SC_K'):
                info['plugin_active'] = '[dg_topk]' in open(log_path).read()
            print(f"  started | KV cache {info['kv_cache_gib']} GiB, {info['kv_cache_tokens']} tokens"
                  + (f" | plugin active: {info['plugin_active']}" if 'plugin_active' in info else ''))
            break
        print('  failed to start; last log lines:\n' + tail(log_path, 8))
        append(dict(config=name, server_max_seqs=m, concurrency=-1, status='failed_to_start',
                    oom='out of memory' in open(log_path).read().lower()))
    if proc is None:
        continue
    # 2) the concurrency sweep on this one server
    try:
        for c in todo:
            if c > server_max:
                continue
            try:
                res = await run_client(c)
                row = dict(config=name, server_max_seqs=server_max, concurrency=c, status='ok', **info, **res)
                print(f"  concurrency {c:3d}: {res['tok_per_s']:7.0f} tok/s | latency p50 {res['lat_p50_s']:5.1f}s | accuracy {res['accuracy']:.3f}")
            except Exception as e:
                row = dict(config=name, server_max_seqs=server_max, concurrency=c, status=f'client_error: {e}')
                print('  client error:', e)
            append(row)
    finally:
        stop_server(proc)
print('done')


## Results

In [ ]:
import matplotlib.pyplot as plt
res = pd.read_csv(CSV)
ok = res[res.status == 'ok']
pd.set_option('display.float_format', '{:.3f}'.format)
cols = [c for c in ['config', 'server_max_seqs', 'concurrency', 'status', 'tok_per_s', 'req_per_s', 'lat_p50_s', 'lat_p90_s', 'accuracy'] if c in res]
print(res[cols].to_string(index=False))

piv = ok.pivot(index='concurrency', columns='config', values='tok_per_s')
for cfg in ('topk64', 'topk64_exact'):
    if {'stock', cfg} <= set(piv.columns):
        piv[f'speedup {cfg}'] = piv[cfg] / piv['stock']
print('\nThroughput (generated tokens / s) by concurrency'); print(piv.to_string())

# Each system at its best on this hardware: the highest throughput it reached, at whatever concurrency it could run.
best = ok.loc[ok.groupby('config').tok_per_s.idxmax(), ['config', 'concurrency', 'tok_per_s', 'lat_p50_s', 'accuracy']]
if 'stock' in set(best.config):
    best['vs stock best'] = best.tok_per_s / float(best.loc[best.config == 'stock', 'tok_per_s'].iloc[0])
print('\nBest throughput per configuration (serving capacity on these GPUs)'); print(best.to_string(index=False))

print('\nLargest --max-num-seqs that started:', ok.groupby('config').server_max_seqs.max().to_dict())
fails = res[res.status == 'failed_to_start']
if len(fails):
    print('Failed starts (config, --max-num-seqs):', list(zip(fails.config, fails.server_max_seqs)))
if 'kv_cache_gib' in ok:
    print('KV-cache memory at startup (GiB per GPU):', ok.groupby('config').kv_cache_gib.first().to_dict())
print('Accuracy (all concurrency levels pooled):', ok.groupby('config').accuracy.mean().round(3).to_dict(),
      '| questions per config:', ok.groupby('config').n.sum().to_dict())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, g in ok.groupby('config'):
    g = g.sort_values('concurrency')
    ax[0].plot(g.concurrency, g.tok_per_s, marker='o', label=name)
    ax[1].plot(g.concurrency, g.lat_p50_s, marker='o', label=name)
ax[0].set(xscale='log', xlabel='concurrent requests', ylabel='generated tokens / s', title='Throughput')
ax[1].set(xscale='log', xlabel='concurrent requests', ylabel='median request latency (s)', title='Latency')
xs = sorted(ok.concurrency.unique())
for a in ax:
    a.set_xticks(xs); a.set_xticklabels(xs); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(f'{OUT}/e2e.png', dpi=150); plt.show()


## Kernel check: vLLM's own kernel vs. our copy

Uses the logits notebook 04 saved (`runs/nb04/step_*.pt`) on one of the chosen GPUs, after all server runs are finished. Both kernels draw their random numbers from the same seed and positions, so argmax and samples should be **identical**, and entropy/probabilities equal up to rounding. The timing ratio should be close to 1.

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = GPUS.split(',')[0]   # this process starts CUDA only now
import torch, hint_bench as hb
saved = hb.load_saved_logits('./runs/nb04/step_*.pt')
if not saved:
    print('no saved logits in runs/nb04/ - skipping the kernel check')
else:
    CL = 256
    L, temps = hb.make_batch(saved, 4, CL, 'cuda')
    a1, s1, e1, _, p1 = hb.row_stats(L, temps, CL, seed=123, probs_dtype=torch.bfloat16)
    a2, s2, e2, p2 = sample_row_stats(L, temps, CL, 123, torch.bfloat16)
    print('argmax  identical:', bool(torch.equal(a1, a2)))
    print('sample  identical:', bool(torch.equal(s1, s2)))
    print('entropy max |diff|:', float((e1 - e2).abs().max()))
    print('probs   max |diff|:', float((p1.float() - p2.float()).abs().max()))
    del L, p1, p2
    rows = []
    V = saved[0][0].shape[-1]
    for B in (1, 8, 32):
        if B * CL * V * 4 * 1.6 > torch.cuda.mem_get_info()[0]:      # logits + bf16 probs must fit
            continue
        L, temps = hb.make_batch(saved, B, CL, 'cuda')
        t_copy = hb.time_cuda(lambda: hb.row_stats(L, temps, CL, probs_dtype=torch.bfloat16))
        t_vllm = hb.time_cuda(lambda: sample_row_stats(L, temps, CL, 0, torch.bfloat16))
        rows.append(dict(B=B, copy_ms=t_copy, vllm_ms=t_vllm, ratio=t_copy / t_vllm))
        del L
    import pandas as pd
    print(pd.DataFrame(rows).to_string(index=False, float_format='{:.3f}'.format))
    torch.cuda.empty_cache()

## How to read it

1. **Check `plugin_active` is True for every `topk64` row.** If not, the plugin did not load in the server (see the log) and the two configs are the same code.
2. **Accuracy** should match between `stock` and `topk64` within noise (100 questions: about ±3 points). Run 250+ questions for the final comparison.
3. **Two comparisons.** (a) Same concurrency: the `speedup` columns show whether each step is faster. (b) Each system at its best: `Best throughput per configuration` compares stock at the largest concurrency it can run with ours at the largest ours can run. If ours fits a larger batch (e.g. 64 where stock stops at 32), `vs stock best` is the serving-capacity gain on the same GPUs.
4. **KV-cache memory at startup:** vLLM profiles one step (including the sampler) and gives the remaining memory to the KV cache. More GiB for `topk64` / `topk64_exact` than for `stock` means the lighter sampler frees memory for more concurrent requests.
5. **Largest `--max-num-seqs` that starts:** vLLM reserves memory for the sampler's per-step tiles; if `topk64` starts at a larger `--max-num-seqs` than `stock`, that is a direct memory result.
6. These numbers include everything (30 layers, attention, scheduling), so the speed-up will be smaller than notebook 08's sampler-only speed-up. That gap is expected and worth reporting.